In [4]:
import torch
from torch import nn
from torch.nn import functional as F
import math

1.PatchEmbed

In [5]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, d_model):
        super().__init__()
        assert img_size % patch_size == 0, "img_size should be divisible by patch_size"

        self.img_size = img_size
        self.patch_size = patch_size

        self.n_patches = (img_size // patch_size) ** 2

        self.projection = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)

        x = x.flatten(x)

        x = x.transpose(1, 2)  # (B, d_model, N) -> (B, N, d_model)

        return x

2.PatchMerging

In [6]:
class PatchMerging(nn.Module):
    # 4C -> 2C
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.reduction = nn.Linear(4 * in_channels, out_channels, bias=False)

        self.norm = nn.LayerNorm(4 * in_channels)

    def forward(self, x, H, W):
        B, L, C = x.shape

        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]

        x = torch.cat([x0, x1, x2, x3], dim=-1)

        x = x.view(B, -1, 4 * C)

        x = self.norm(x)

        x = self.reduction(x)

        return x

3.WindowAttention

In [7]:
class WindowAttention(nn.Module):
    def __init__(self, d_model, n_heads, window_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model should be divisible by n_heads"

        self.d_model = d_model
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.window_size = window_size

        # 注意力投影
        self.W_qkv = nn.Linear(d_model, d_model * 3)
        self.fc = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.softmax = nn.Softmax(dim=-1)

        # 相对位置偏置表
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), n_heads)
        )

        # 预计算相对位置索引
        self.register_buffer("relative_position_index", self._get_relative_position_index())

        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

    def _get_relative_position_index(self):
        M = self.window_size

        coords = torch.arange(M)

        coords_h, coords_w = torch.meshgrid(coords, coords, indexing="ij")

        coords = torch.stack([coords_h, coords_w])

        coords_flat = coords.flatten(1)

        # 计算所有token对的相对坐标
        # (2, 49, 1) - (2, 1, 49) -> (2, 49, 49)
        relative_coords = coords_flat[:, :, None] - coords_flat[:, None, :]

        relative_coords = relative_coords.permute(1, 2, 0)  # (49, 49, 2)

        # 将负数偏移变成非负索引
        relative_coords[:, :, 0] += M - 1
        relative_coords[:, :, 1] += M - 1

        # 将二维索引映射到一维索引  row * 列数 + col
        relative_position_index = relative_coords[:, :, 0] * (2 * M -1) + relative_coords[:, :, 1]

        return relative_position_index

    def forward(self, x, mask=None):
        B_, N, C = x.shape

        qkv = self.W_qkv(x).view(B_, N, 3, self.n_heads, self.d_k)
        qkv = qkv.permute(2, 0, 4, 1, 3)

        Q, K, V = qkv[0], qkv[1], qkv[2]

        attn = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)

        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(
            self.window_size ** 2,
            self.window_size ** 2,
            -1
        )

        relative_position_bias = relative_position_bias.permute(2, 0, 1).unsqueeze(0)

        attn = attn + relative_position_bias

        if mask is not None:
            # 窗口数
            nW = mask.shape[0]
            # (batch, 窗口数, 头数, 窗口维度, 窗口维度)
            attn = attn.view(B_ // nW, nW, self.n_heads, N, N)
            
            attn = attn + mask.unsqueeze(1).unsqueeze(0)

            attn = attn.view(-1, self.n_heads, N, N)

        attn = self.softmax(attn)
        attn = self.dropout(attn)

        out = (attn @ V).transpose(1, 2).contiguous().view(B_, N, C)

        out = self.fc(out)

        out = self.dropout(out)

        return out

SwinTransformerBlock

In [ ]:
class SwinTransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, window_size, shift_size, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.window_size = window_size
        self.shift_size = shift_size

        self.norm1 = nn.LayerNorm(d_model)
        self.attn = WindowAttention(d_model, n_heads, window_size, dropout)

        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def _window_partition(self, x, H, W):
        """
        将特征图切成 M × M的不重叠窗口
        (B, H, W, C) -> (B, H/M, M, W/M, M, C) -> (B, H/W, W/M, M, M, C) -> (B*num_windows, M, M, C)
        """
        M = self.window_size
        B = x.shape[0]

        x = x.view(B, H // W, M, W // M, M, -1)

        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()

        x = x.view(B, -1, M, M, x.shape[-1])

        x = x.view(-1, M, M, x.shape[-1])

        return x

    def _window_reverse(self, windows, H, W):
        '''
        _window_partition 的逆操作，把窗口拼回原图
        (B*num_windows, M, M, C) -> (B, H, W, C)
        '''

        M = self.window_size
        C = windows.shape[-1]

        # 算batch大小
        # windows: (B*nW, window_size, window_size, C)
        B = windows.shape[0] // ((H // M) * (W // M))

        x = windows.view(B, H // M, W // M, M, M, C)

        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()

        x = x.view(B, H, W, C)

        return x

    def _get_attn_mask(self, H, W, device):
        M = self.window_size  # 7
        shift = self.shift_size  # 3

        # 创建一张全零的编号图(1, H, W, 1)
        img_mask = torch.zeros((1, H, W, 1), device=device)

        cnt = 0
        h_slices = (slice(0, -M), slice(-M, -shift), slice(-shift, None))

        w_slices = (slice(0, -M), slice(-M, -shift), slice(-shift, None))

        # 对像素进行编号
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        # 把编号图切成窗口
        mask_windows = self._window_partition(img_mask, H, W)
        mask_windows = mask_windows.view(-1, M * M)

        '''
        mask_windows.unsqueeze(1): (nW, 1, 49)
        mask_windows.unsqueeze(2): (nW, 49, 1)
        利用广播减法 -> (nW, 49, 49)
        编号相同：差=0 -> mask=0 (不屏蔽)
        编号不相同：差≠0 -> mask=-100(屏蔽，softmax后≈0)
        '''
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0))
        attn_mask = attn_mask.masked_fill(attn_mask == 0, float(0.0))

        return attn_mask

    def forward(self, x, H, W):
        # pre-norm + shortcut
        shortcut = x   # 保留残差连接
        x = self.norm1(x)

        # 输入 x 形状 (B, H×W, C)
        x = x.view(x.shape[0], H, W, -1)

        # 循环移位（只 SW-MSA 做）
        if self.shift_size > 0:
            # torch.roll（循环位移）—— 超出边界的元素从另一边绕回来
            # shifts=()表示在指定的dims=(),各移动指定的步数
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        # 窗口切分
        x_windows = self._window_partition(shifted_x, H, W)
        x_windows = x_windows.view(x_windows.shape[0], self.window_size **2, -1)

        # 生成mask（只 SW-MSA 做）
        attn_mask = self._get_attn_mask(H, W, x.device) if self.shift_size > 0 else None

        # 窗口内注意力
        attn_out = self.attn(x_windows, attn_mask)

        # 恢复空间形状
        attn_out = attn_out.view(-1, self.window_size, self.window_size, self.d_model)
        shifted_x = self._window_reverse(attn_out, H, W)

        # 逆向循环移位（只 SW-MSA 做）
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        # 恢复序列 + 残差
        x = x.view(x.shape[0], H*W, -1)
        x = shortcut + x

        # MLP
        shortcut = x
        x = self.norm2(x)
        x = shortcut + self.ffn(x)

        return x

BasicLayer(一个Stage)

In [ ]:
'''
Swin 有 4 个 Stage，每个 Stage 的结构都一样
    1. 开头一个 PatchMerging（Stage 1 例外，因为 PatchEmbed 已经做好）
    2. 偶数个 SwinTransformerBlock，W-MSA 和 SW-MSA 交替，感受野逐步变大
'''

class BasicLayer(nn.Module):
    '''一个 Stage = 降采样（可选） + N 个交替的 Swin Block'''
    def __init__(self, d_model, depth, n_heads, window_size, d_ff, dropout=0.1, downsample=None):
        super().__init__()
        # downsample 是 PatchMerging 实例，Stage 1 时传 None
        self.downsample = downsample

        # 构建 block 列表
        # W-MSA 和 SW-MSA 交替
        self.blocks = nn.ModuleList([
            
        ])